# LangGraph Practice # 5


### 구성도

![구성도]()

In [1]:
import os,sys
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('utils'), '..')))
from module.utils import * 
from module.prompt import * 
from module.custom_model import *
from typing import List

In [2]:
strt_langsmith('practive_5')

LangSmith 추적을 시작합니다.
[프로젝트명]
practive_5


In [3]:
from typing import TypedDict, Annotated, List, Literal
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage,ToolMessage

In [4]:
class GradeDocument(BaseModel):
    """
        you can says 'yes' or 'no'
        If this document is relevance about question, you say 'yes' and otherwise 'no'
    """
    answer:Literal['yes','no'] = Field(
        ...,
        description="Documents are relevant to the question, 'yes' or 'no'"
        
    )

loader = get_pdf_loader()
splitter = get_text_splitter()
docs = get_docs(loader,splitter)
embedding = get_embedding()
retriever = get_retriever(docs,embedding,k=7)



In [5]:
class State(TypedDict):
    question : Annotated[str,'user input question']   # 사용자 질의 or requeustion 질의
    messages : Annotated[list,add_messages]  # llm , retriever 등에서 생성된 데이터 원본 
    answer : Annotated[str,'llm final answer'] # llm 이 생성한 최종 대답
    documents : Annotated[list,'use retriever'] # retreiver 정제한 데이터

In [6]:
def retrieve(state : State)-> State:
    """ 
        사용자의 question을 받아 vectorestore에서 retriever 하는 과정
        3개의 chunk를 판단함
    """
    question = state['question']
    documents = retriever.invoke(question)
    messages = convert_docs_str(documents)
    return State({'messages':messages})

def grade_document(state : State)->State:
    """
        document의 관련성 여부를 판단하여 관련성 있는 파일만 추출하여 수집하는 단계
    """
    prompt = get_prompt_grade()
    llm = get_gemini(temperature=0.5)
    llm_with_grade=llm.with_structured_output(GradeDocument)
    chain = prompt | llm_with_grade
    origin_docs = convert_str_to_docs(state['messages'])
    filter_docs = []
    for doc in origin_docs:
        response = chain.invoke({'question':state['question'],'document':doc.page_content})
        if response.answer == 'yes':
            filter_docs.append(doc)
    return State({'documents':filter_docs})

def query_re_write(state : State)-> State:
    """
        grade_docuemtns 결과 docs 문서 수가 3이하인경우 `question` 변경하여 재요청
    """
    chain = get_prompt_re_write() | get_gemini(temperature=0.7) | StrOutputParser()
    new_question = chain.invoke({'question':state['question']})
    return State({'question':new_question})


def generate(state : State)->State:
    """
        grade_docuemtns 결과 docs 문서 수가 3이상인 경우 문서 생성
    """
    chain = get_prompt_rag() | get_gemini(temperature=0.7) | StrOutputParser()
    answer = chain.invoke({'question':state['question'],'context':state['documents']})
    return State({'answer':answer})



In [7]:
def is_grade(state : State)->Literal['generate','query_re_write']:
    if len(state['documents'])  >= 3:
        return 'generate'
    else: 
        return 'query_re_write'


def is_hallu_relevance(state : State)->Literal['generate','query_re_write',END]:
    chain =  get_relevant(llm=get_gemini(),target='retrieval-answer') 
    hallucination = chain.invoke({'answer':state['answer'],'context':state['documents']})
    if hallucination.score == 'yes' : # document 와 answer가 연관 있다
        _chain =  get_relevant(llm=get_gemini(),target='question-answer') 
        relevant = _chain.invoke({'question':state['question'],'answer':state['answer']})
        if relevant.score =='yes':
            return END
        else:
            return 'query_re_write'

    else : # document 와 answer가 연관 없다
        return 'generate'

In [8]:
state_graph = StateGraph(State)
state_graph.add_node('retrieve',retrieve)
state_graph.add_node('grade_document',grade_document)
state_graph.add_node('query_re_write',query_re_write)
state_graph.add_node('generate',generate)

state_graph.add_edge(START,'retrieve')
state_graph.add_edge('retrieve','grade_document')
state_graph.add_conditional_edges(
    source='grade_document',
    path=is_grade,
    path_map=['generate','query_re_write']
)
state_graph.add_conditional_edges(
    source='generate',
    path=is_hallu_relevance
)

state_graph.add_edge('query_re_write','retrieve')

memory = get_check_pointer()

graph = state_graph.compile(checkpointer=memory)



In [9]:
# visualize_graph(graph)
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	grade_document(grade_document)
	query_re_write(query_re_write)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	generate -.-> __end__;
	generate -.-> query_re_write;
	grade_document -.-> generate;
	grade_document -.-> query_re_write;
	query_re_write --> retrieve;
	retrieve --> grade_document;
	grade_document -.-> __end__;
	generate -.-> generate;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [12]:
uuid = get_random_uuid()
config = get_runnable_config(recursion_limit=10,thread_id=uuid)

In [ ]:
inputs ={'question':'오늘의 뉴스'}
stream_graph(graph,inputs,config)


🔄 Node: grade_document 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: generate 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
삼성전자는 자체 개발 생성 AI 모델인 '삼성 가우스'를 공개했습니다. 이 모델은 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 세 가지 모델로 구성됩니다.

* ../data/SPRI_AI_Brief_2023년12월호_F.pdf (page 13)
* ../data/SPRI_AI_Brief_2023년12월호_F.pdf (page 2)

KeyboardInterrupt: 

In [ ]:
invoke_graph(graph,inputs,config)